In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))

import torch
from torch.utils.data import DataLoader

from src.cache import get_image_features
from src.dataset import OCLDataset
from src.model import load_model, encode_text, encode_image
from src.prompts import build_category_prompts, build_attribute_prompts, build_affordance_prompts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cache_dir = Path(os.environ.get("OCL_CACHE_DIR", data_root / "cache"))
print(device)

- ViT-B/32
- prompt:
  - `a photo of a {category}`
  - `a photo of an object that is {attribute}`
  - `a photo of an object that can be used to {affordance['word'][0]}`

In [ ]:
model, preprocess, weights_sha256 = load_model("ViT-B/32", device, os.environ.get("OCL_WEIGHTS_DIR"))
dataset = OCLDataset(data_root, resource_root, preprocess)
test_loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0, drop_last=False)

In [ ]:
category_prompts = build_category_prompts(dataset.categories)

print(category_prompts)

category_features = encode_text(
    model,
    category_prompts,
    device
)

print(category_features.shape)
print(category_features.norm(dim=-1))

In [ ]:
attribute_prompts = build_attribute_prompts(dataset.attributes)
print(attribute_prompts[0:5])

affordance_prompts = build_affordance_prompts(dataset.affordances, mode="word")

print(affordance_prompts[0:5])

attribute_features = encode_text(
    model,
    attribute_prompts,
    device
)


affordance_features = encode_text(
    model,
    affordance_prompts,
    device
)
print(attribute_features.shape)

print(affordance_features.shape)

In [ ]:
batch = next(iter(test_loader))

image_features = encode_image(
    model,
    batch["image"],
    device
)

print(image_features.shape)

category_scores = image_features @ category_features.T

print(category_scores.shape)
print(category_scores)

In [ ]:
predictions = category_scores.argmax(dim=1)

for pred, target in zip(
    predictions,
    batch["category"]
):
    print(
        "Prediction:", dataset.categories[pred.item()],
        "| Ground truth:", dataset.categories[target.item()]
    )

In [ ]:
print(dataset.affordances[2])
print(dataset.affordances[3])

In [ ]:
from collections import Counter

prompt_counts = Counter(affordance_prompts)

duplicates = {
    prompt: count
    for prompt, count in prompt_counts.items()
    if count > 1
}

print(duplicates)
print("Unique prompts:", len(prompt_counts))

In [ ]:
from src.metrics import topk_accuracy

for k in [1, 3, 5]:
    acc = topk_accuracy(
        category_scores,
        batch["category"].to(device),
        k=k
    )

    print(f"Top-{k} Accuracy: {acc:.2%}")

In [ ]:
results, cache_key, cache_hit = get_image_features(
    model, test_loader, device, weights_sha256, use_cache=True, cache_dir=cache_dir
)

for key, value in results.items():
    print(key, value.shape)

In [ ]:
print("Shared image cache key:", cache_key)
print("Reused cached image features:", cache_hit)

In [ ]:
image_features = results["features"]

category_scores = (
    image_features @ category_features.cpu().T
)

attribute_scores = (
    image_features @ attribute_features.cpu().T
)
affordance_scores = (
    image_features @ affordance_features.cpu().T
)
print(category_scores.shape)
print(attribute_scores.shape)
print(affordance_scores.shape)


In [ ]:
from src.metrics import (
    topk_accuracy,
    mean_average_precision
)

for k in [1, 3, 5]:

    acc = topk_accuracy(
        category_scores,
        results["category"],
        k=k
    )

    print(f"Top-{k} Accuracy: {acc:.2%}")

In [ ]:
attr_map, attr_ap = mean_average_precision(
    attribute_scores,
    results["attribute"]
)

print("Attribute mAP:", "undefined" if attr_map is None else f"{attr_map:.2%}")

In [ ]:
affo_map,affo_ap = mean_average_precision(
    affordance_scores,
    results["affordance"]
)
print("Affordance mAP:", "undefined" if affo_map is None else f"{affo_map:.2%}")

In [ ]:
import numpy as np

print(
    "Valid attribute classes:",
    np.isfinite(attr_ap).sum(),
    "/",
    len(attr_ap)
)

In [ ]:
attr_prevalence = (
    results["attribute"].float().mean(dim=0).numpy()
)

aff_prevalence = (
    results["affordance"].float().mean(dim=0).numpy()
)

attr_baseline = attr_prevalence[
    attr_prevalence > 0
].mean()

aff_baseline = aff_prevalence[
    aff_prevalence > 0
].mean()

print(f"Attribute baseline: {attr_baseline:.2%}")
print(f"Affordance baseline: {aff_baseline:.2%}")

In [ ]:
print(
    "Attributes per object:",
    results["attribute"].sum(dim=1).float().mean().item()
)

print(
    "Affordances per object:",
    results["affordance"].sum(dim=1).float().mean().item()
)

In [ ]:
missing_attr = [
    dataset.attributes[i]
    for i in range(len(attr_ap))
    if np.isnan(attr_ap[i])
]

print(missing_attr)

In [ ]:
valid_indices = np.where(np.isfinite(attr_ap))[0]

sorted_indices = valid_indices[
    np.argsort(attr_ap[valid_indices])[::-1]
]

print("Best 10 attributes:")

for i in sorted_indices[:10]:
    print(
        f"{dataset.attributes[i]:20s}",
        f"AP: {attr_ap[i]:.2%}",
        f"Prevalence: {attr_prevalence[i]:.2%}"
    )

print("\nWorst 10 attributes:")

for i in sorted_indices[-10:]:
    print(
        f"{dataset.attributes[i]:20s}",
        f"AP: {attr_ap[i]:.2%}",
        f"Prevalence: {attr_prevalence[i]:.2%}"
    )